In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/18 14:49:54 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/18 14:49:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0ed5780a-6932-4b9c-a950-38fd019e28e3;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 79ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = 'd3282a32-5edc-4d85-b48c-e0ea38d9a4de'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-18 14:49:59,970 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-18 14:49:59,970 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-18 14:49:59,973 | INFO | break_analysis.agent | Analyzing break case | case_id=551eeed4 | workflow_run_id=d3282a32 | topology=MANY_TO_ONE | records=3
2026-09-18 14:49:59,974 | INFO | break_analysis.agent | Invoking LLM | case_id=551eeed4 | round=1
2026-09-18 14:50:17,210 | INFO | break_analysis.agent | LLM invoked | case_id=551eeed4 | round=1 | tool_calls=4 | input_tokens=1091 | output_tokens=203 | total_tokens=1294 | prompt_eval_count=1091 | eval_count=203 | load_ms=9964 | prompt_eval_ms=862 | eval_ms=6391 | total_ms=17225 | duration_ms=17236
2026-09-18 14:50:17,211 | INFO | break_analysis.agent | Tool round | case_id=551eeed4 | round=1 | tool_calls=4
2026-09-18 14:50:17,437 | INFO | break_analysis.agent | Tool invoked | case_id=551eeed4 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "4100000", "business_dt": "2026-03-31"} | duration_ms=226
2026-09-18 14:50:17,553 | INFO | break_analysis.agent | Tool invoked | case_id=551eeed4 | tool=validate_se

BreakAnalysisResult(case_id=UUID('551eeed4-19c8-48f4-9c3a-b0392bf29bac'), recon_result_ids=(UUID('bf31b11b-2740-4649-8543-891f56e01821'), UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), UUID('828d5971-42c9-40dc-b0e7-96e5faf562af')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, findings=(BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), segment_type=<GLSegmentType.ACCOUNT: 'GL_ACCOUNT'>, segment_value='4100000', explanation='Registry record is missing.'), BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), segment_type=<GLSegmentType.SUB_ACCOUNT: 'GL_SUB_ACCOUNT'>, segment_value='004000', explanation='Registry record is inactive.'), BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('828d5971-42c9-40dc-b0e7-96e5faf562af'), 

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-18 14:51:13,620 | INFO | break_analysis.agent | Analyzing break case | case_id=c160b8cf | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-18 14:51:13,621 | INFO | break_analysis.agent | Invoking LLM | case_id=c160b8cf | round=1
2026-09-18 14:51:16,037 | INFO | break_analysis.agent | LLM invoked | case_id=c160b8cf | round=1 | tool_calls=1 | input_tokens=842 | output_tokens=50 | total_tokens=892 | prompt_eval_count=842 | eval_count=50 | load_ms=121 | prompt_eval_ms=291 | eval_ms=1727 | total_ms=2414 | duration_ms=2416
2026-09-18 14:51:16,038 | INFO | break_analysis.agent | Tool round | case_id=c160b8cf | round=1 | tool_calls=1
2026-09-18 14:51:16,128 | INFO | break_analysis.agent | Tool invoked | case_id=c160b8cf | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "210000", "business_dt": "2026-03-31"} | duration_ms=90
2026-09-18 14:51:16,129 | INFO | break_analysis.agent | Invoking LLM | case_id=c160b8cf | round=2
2026-09-18 14:51:17

BreakAnalysisResult(case_id=UUID('c160b8cf-839c-4ceb-a79c-c47048696001'), recon_result_ids=(UUID('8b947b0d-0d8b-4b7c-925c-2c0af4628151'), UUID('a8887fa1-74d8-46a2-878d-ebf5de37e8db')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, findings=(BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('a8887fa1-74d8-46a2-878d-ebf5de37e8db'), segment_type=<GLSegmentType.ACCOUNT: 'GL_ACCOUNT'>, segment_value='210000', explanation="Registry record exists but is inactive (status='I')"),), explanation="The discrepancy is fully explained by an invalid GL_ACCOUNT segment value '210000' in the investigation record. The registry record for this segment exists but is marked as inactive, preventing proper reconciliation.")

In [9]:
break_case = break_cases[2]

agent.analyze(break_case)

2026-09-18 14:51:38,636 | INFO | break_analysis.agent | Analyzing break case | case_id=37923656 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-18 14:51:38,637 | INFO | break_analysis.agent | Invoking LLM | case_id=37923656 | round=1
2026-09-18 14:51:44,071 | INFO | break_analysis.agent | LLM invoked | case_id=37923656 | round=1 | tool_calls=3 | input_tokens=846 | output_tokens=130 | total_tokens=976 | prompt_eval_count=846 | eval_count=130 | load_ms=123 | prompt_eval_ms=305 | eval_ms=4853 | total_ms=5432 | duration_ms=5434
2026-09-18 14:51:44,071 | INFO | break_analysis.agent | Tool round | case_id=37923656 | round=1 | tool_calls=3
2026-09-18 14:51:44,165 | INFO | break_analysis.agent | Tool invoked | case_id=37923656 | tool=validate_segment | args={"business_dt": "2026-03-31", "segment_type": "GL_ACCOUNT", "segment_value": ""} | duration_ms=94
2026-09-18 14:51:44,252 | INFO | break_analysis.agent | Tool invoked | case_id=37923656 | tool=validate_segment | args={"

BreakAnalysisResult(case_id=UUID('37923656-ed04-46f1-a1fb-7b6d193a200c'), recon_result_ids=(UUID('a0498239-ac79-4748-8ef6-6148cf9151b3'), UUID('4424b323-4067-433d-a23f-b534b6cf4a2e')), status=<BreakAnalysisStatus.UNEXPLAINED: 'UNEXPLAINED'>, findings=(), explanation='The investigation record contains blank values for GL_ACCOUNT, GL_SUB_ACCOUNT, and GL_PRODUCT segments, which do not support the REGISTRY_INVALID_SEGMENT hypothesis. No nonblank segments were validated against the registry, and no other evidence indicates invalid registry entries.')

In [10]:
# spark.stop()